# 06 — Minimal Loader Path

Verify the minimal bike-sharing contract:

* `DataLoaderMockMinimal` emits only `df_stations` + `df_trips`; every other
  optional source field is `None`.
* `DataLoaderGraph` synthesizes the skeleton that `build_model` needs
  (planning horizon from trip dates, commodity categories from
  `rideable_type` or a fallback, station-to-station edge rule, no depots,
  no resources, no capacity tables).
* `build_model` derives `periods`, `facility_roles`, `demand`, `supply`,
  and seeds `inventory_initial` from first-day outflow.
* An `Environment` with `DemandPhase + ArrivalsPhase` runs end-to-end on
  the resulting `ResolvedModelData` and produces populated log tables.

In [ ]:
from gbp.loaders import DataLoaderMockMinimal

source = DataLoaderMockMinimal({'n_stations': 6, 'n_trips': 100, 'n_days': 3})
source.load_data()

populated = {
    name: getattr(source, name) is not None
    for name in (
        'df_stations', 'df_trips', 'df_depots', 'df_resources',
        'df_station_capacities', 'df_depot_capacities', 'df_resource_capacities',
        'timestamps', 'df_inventory_ts', 'df_telemetry_ts',
        'df_station_costs', 'df_depot_costs', 'df_truck_rates',
    )
}
populated

In [ ]:
from gbp.loaders import DataLoaderGraph, GraphLoaderConfig

loader = DataLoaderGraph(source, GraphLoaderConfig())
raw = loader.load()

raw_summary = {
    'facilities': len(raw.facilities) if raw.facilities is not None else 0,
    'commodity_categories': list(raw.commodity_categories['commodity_category_id']) if raw.commodity_categories is not None else [],
    'edge_rules': len(raw.edge_rules) if raw.edge_rules is not None else 0,
    'observed_flow_rows': len(raw.observed_flow) if raw.observed_flow is not None else 0,
    'resource_categories_is_none': raw.resource_categories is None,
    'resource_fleet_is_none': raw.resource_fleet is None,
    'observed_inventory_is_none': raw.observed_inventory is None,
}
raw_summary

In [ ]:
from gbp.build.pipeline import build_model

resolved = build_model(raw)
resolved.build_report.derivations

In [ ]:
# inventory_initial was seeded from first-day outflow (no observed_inventory telemetry).
resolved.inventory_initial

In [ ]:
resolved_summary = {
    'periods': len(resolved.periods),
    'facilities': len(resolved.facilities),
    'edges': 0 if resolved.edges is None else len(resolved.edges),
    'demand': 0 if resolved.demand is None else len(resolved.demand),
    'supply': 0 if resolved.supply is None else len(resolved.supply),
    'inventory_initial': len(resolved.inventory_initial),
    'fleet_capacity_is_none': resolved.fleet_capacity is None,
    'resource_spines_is_none': resolved.resource_spines is None,
}
resolved_summary

In [ ]:
from gbp.consumers.simulator.built_in_phases import ArrivalsPhase, DemandPhase
from gbp.consumers.simulator.config import EnvironmentConfig
from gbp.consumers.simulator.engine import Environment

env = Environment(
    resolved,
    EnvironmentConfig(phases=[DemandPhase(), ArrivalsPhase()]),
)
env.run()

log_tables = env.log.to_dataframes()
log_summary = {name: len(df) for name, df in log_tables.items()}
log_summary

In [ ]:
log_tables['simulation_inventory_log'].head(20)